## **LAB - Testes e Análises dos Dados de Medidores**

#### Esse notebook visa realizar testes e análises para o desenvolvimento do problema envolvendo as imagens dos medidores tirados pelos leituristas da Neroenergia.

- Configurações iniciais

In [ ]:
# Bibliotecas
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import glob
import os

# Pandas configs
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)

- **Tópico 1 -** Análise dos arquivos `CSVs`

In [15]:
# Analisando estrutura dos dados CSVs
sheets = glob.glob('../data/sheets/*.csv')
df = pd.concat([pd.read_csv(sheet, sep=';') for sheet in sheets])

display(df.head())
df.info()

,Numero do medidor,Posicao do medidor lida,Nota de Leitura Atual,Foto do medidor
0,3213492267,6521,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000012852988478_000.jpg
1,40482962,9729,L131,PSP_EXTRATLEITIMPL_030726_0121_20000000003053017019_000.jpg
2,30501853,24788,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000006502995976_000.jpg
3,82861661,20512,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000012552976989_000.jpg
4,3221134515,1698,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000017102988986_000.jpg


<class 'pandas.core.frame.DataFrame'>
Index: 13669 entries, 0 to 3259
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Numero do medidor        13669 non-null  object
 1   Posicao do medidor lida  13669 non-null  int64 
 2   Nota de Leitura Atual    4560 non-null   object
 3   Foto do medidor          11474 non-null  object
dtypes: int64(1), object(3)
memory usage: 533.9+ KB


Após análise inicial dos dados, é possível ver que as colunas `Numero do medidor` e `Posicao do medidor lida` estão todas preenchidas. Já as demais colunas estão com uma boa quantidade de valores nulos. O próximo passo é identificar mais detalhadamente esses valores.

In [17]:
# Verificando colunas com valores nulos
print("Verificação de valores nulos [Nota de Leitura Atual]:", df["Nota de Leitura Atual"].isnull().sum())
print("Verificação de valores nulos [Foto do medidor]:", df["Foto do medidor"].isnull().sum())

Verificação de valores nulos [Nota de Leitura Atual]: 9109
Verificação de valores nulos [Foto do medidor]: 2195


In [26]:
# Identificando os valores nulos com mais detalhe
print("Nota de Leitura Atual - Valores nulos:")
display(df[df["Nota de Leitura Atual"].isnull()].head(15))

print("Foto do medidor - Valores nulos:")
df[df["Foto do medidor"].isnull()].head(15)

Nota de Leitura Atual - Valores nulos:


,Numero do medidor,Posicao do medidor lida,Nota de Leitura Atual,Foto do medidor
0,3213492267,6521,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000012852988478_000.jpg
2,30501853,24788,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000006502995976_000.jpg
3,82861661,20512,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000012552976989_000.jpg
4,3221134515,1698,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000017102988986_000.jpg
5,82806814,14247,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000018003004648_000.jpg
8,3220820563,606,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000000202980515_000.jpg
9,3172755150,23591,NaN,NaN
10,L23637,29997,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000006602993796_000.jpg
12,3201867893,32174,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000015502982528_000.jpg
15,50628191,85044,NaN,PSP_EXTRATLEITIMPL_030726_0121_20000000012053008152_000.jpg


Foto do medidor - Valores nulos:


,Numero do medidor,Posicao do medidor lida,Nota de Leitura Atual,Foto do medidor
9,3172755150,23591,NaN,NaN
19,IC60087,38521,NaN,NaN
20,30497350,17226,NaN,NaN
31,3214285281,117062,NaN,NaN
32,3181710895,383632,NaN,NaN
43,6252249727/3191280490,821,NaN,NaN
44,3241205397,19732,NaN,NaN
54,3161420840,233093,NaN,NaN
58,3180658687,777398,NaN,NaN
66,3181692820,298285,NaN,NaN


A ausência de `Foto do medidor` parece explicar parte dos nulos em `Nota de Leitura Atual` (sem foto, não há como extrair a nota), mas não toda: há linhas com foto presente e nota nula (ex.: índices 0, 2, 3, 4). Antes de seguir, vale checar duplicatas e a distribuição das colunas.

In [ ]:
# Verificando duplicatas
print("Linhas totalmente duplicadas:", df.duplicated().sum())
print("Numero do medidor duplicado:", df["Numero do medidor"].duplicated().sum())
print("Numero do medidor - valores unicos:", df["Numero do medidor"].nunique(), "de", len(df), "linhas")

Apenas 1 linha totalmente duplicada e 12 medidores repetidos em 13669 registros (13657 valores únicos) — duplicação é praticamente irrelevante nesta base. O próximo passo é olhar a distribuição das colunas `Nota de Leitura Atual` e `Posicao do medidor lida`.

In [ ]:
# Distribuição de "Nota de Leitura Atual"
notas = df["Nota de Leitura Atual"].value_counts()
display(notas)

plt.figure(figsize=(10, 4))
plt.bar(notas.index, notas.values, color="#4C72B0")
plt.title("Distribuição da Nota de Leitura Atual")
plt.xlabel("Nota de Leitura")
plt.ylabel("Quantidade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

`T181` domina amplamente as notas preenchidas (2595 de 4560 não nulas), seguida de `P111` e `L101`. As demais categorias somam poucas ocorrências, indicando forte desbalanceamento entre as notas de leitura.

In [ ]:
# Distribuição de "Posicao do medidor lida"
display(df["Posicao do medidor lida"].describe())

plt.figure(figsize=(10, 4))
plt.hist(df["Posicao do medidor lida"], bins=50, color="#4C72B0")
plt.title("Distribuição da Posição do Medidor Lida")
plt.xlabel("Posição do medidor lida")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()

# Possíveis valores sentinela/outliers
print("Registros com valor 0:", (df["Posicao do medidor lida"] == 0).sum())
print("Registros com valor 999999 (possível sentinela):", (df["Posicao do medidor lida"] == 999999).sum())

A distribuição é bastante assimétrica à direita (mediana ~13.577 vs. média ~43.411), com uma cauda longa até o máximo teórico de leitura. Há 347 registros com posição igual a 0 e 15 registros com valor 999999, que parecem ser valores sentinela/erro de leitura e merecem tratamento (remoção ou flag) antes de qualquer modelagem.

- **Tópico 2 -** Correspondência entre os `CSVs` e as imagens em disco

In [ ]:
# Comparando nomes de arquivo citados no CSV com os arquivos existentes em disco
image_paths = glob.glob('../data/meter_images/*/*.jpg')
print("Total de imagens em disco:", len(image_paths))

disk_names = set(os.path.basename(p) for p in image_paths)
csv_names = set(df["Foto do medidor"].dropna())

print("Nomes de imagem citados no CSV:", len(csv_names))
print("Citados no CSV e presentes em disco:", len(csv_names & disk_names))
print("Citados no CSV mas AUSENTES em disco:", len(csv_names - disk_names))
print("Presentes em disco mas NAO citados no CSV (orfãs):", len(disk_names - csv_names))

Todas as imagens citadas no CSV (11.474) existem em disco — não há referências quebradas. Por outro lado, das 12.340 imagens em disco, 866 não são citadas em nenhum CSV (imagens "órfãs"), o que sugere leituras feitas que não geraram registro na planilha, ou fotos extras/descartadas pelos leituristas.

- **Tópico 3 -** Análise exploratória das imagens dos medidores

In [ ]:
# Verificando dimensões e tamanho de arquivo das imagens, por pasta de extração
folders = sorted(glob.glob('../data/meter_images/*'))

for folder in folders:
    imgs = glob.glob(os.path.join(folder, '*.jpg'))[:100]
    dims = set()
    file_sizes_kb = []
    for p in imgs:
        img = cv2.imread(p)
        dims.add(img.shape[:2])  # (altura, largura)
        file_sizes_kb.append(os.path.getsize(p) / 1024)

    print(f"{os.path.basename(folder)}: {len(glob.glob(os.path.join(folder, '*.jpg')))} imagens")
    print(f"  Dimensões (altura, largura) na amostra: {dims}")
    print(f"  Tamanho de arquivo (KB) - min/media/max: {min(file_sizes_kb):.1f} / {np.mean(file_sizes_kb):.1f} / {max(file_sizes_kb):.1f}")
    print()

Todas as pastas seguem o mesmo padrão: imagens de 480x360 pixels (altura x largura), com tamanho de arquivo pequeno (na casa de dezenas de KB). Isso indica um pipeline de captura padronizado entre as 4 extrações. Vale visualizar uma amostra das imagens para inspecionar qualidade, enquadramento e legibilidade do display do medidor.

In [ ]:
# Visualizando uma amostra aleatória de imagens
rng = np.random.default_rng(42)
sample_paths = rng.choice(image_paths, size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, path in zip(axes.flat, sample_paths):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(os.path.basename(path), fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.show()

#### Resumo da análise exploratória

- **Qualidade dos dados**: duplicação é irrelevante (1 linha e 12 medidores repetidos em 13.669); `Nota de Leitura Atual` tem 66,6% de nulos e é fortemente desbalanceada (`T181` domina); `Posicao do medidor lida` tem cauda longa e valores sentinela suspeitos (0 e 999999) que precisam de tratamento.
- **Imagens**: todas as 11.474 imagens citadas no CSV existem em disco (nenhuma referência quebrada); 866 imagens em disco não são citadas em nenhum CSV; todas seguem o mesmo padrão de captura (480x360 px).
- **Próximos passos sugeridos**: (1) decidir tratamento para valores sentinela em `Posicao do medidor lida`; (2) investigar por que 63,3% das linhas com foto presente ainda têm `Nota de Leitura Atual` nula (provavelmente leitura ainda não processada/validada, não falta de imagem); (3) usar a amostra de imagens para validar o enquadramento do display antes de definir a estratégia de extração (OCR/CV) dos dígitos do medidor.